# Similarity Retrieval Test Notebook

This notebook tests vector-space information retrieval for resume-job matching.

In [1]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == 'tests' else Path.cwd()
src_path = project_root / 'src'
dataset_dir = project_root / 'Dataset'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from data_loader import load_train_test_data
from preprocessing import preprocess_dataframe
from similarity_retrieval import (
    create_retrieval_vectorizer,
    build_resume_vector_space,
    transform_job_description,
    compute_similarity_scores,
    rank_resumes_by_similarity,
    get_nonzero_tfidf_terms,
    explain_term_products,
    retrieval_report,
)

print('Project root :', project_root)
print('Dataset path :', dataset_dir)
print('Train exists :', (dataset_dir / 'train.csv').exists())
print('Test exists  :', (dataset_dir / 'test.csv').exists())

Project root : c:\Users\POOJA\OneDrive\Desktop\project
Dataset path : c:\Users\POOJA\OneDrive\Desktop\project\Dataset
Train exists : True
Test exists  : True


## 1) Load and preprocess dataset

In [2]:
train_df, test_df = load_train_test_data(dataset_dir)
train_processed = preprocess_dataframe(train_df, text_column='text')

print('Train shape:', train_processed.shape)
display(train_processed.head(2))

Train shape: (1986, 2)


,text,label
0,engin consult profession summari deliv valu pr...,CONSULTANT
1,carpent summari carpent foreman posit effect u...,CONSTRUCTION


## 2) Build vector space and compute similarity

In [3]:
resume_texts = train_processed['text'].head(10).tolist()
job_description = 'Looking for a data science profile with python, machine learning, and statistics.'

vectorizer = create_retrieval_vectorizer(max_features=10000, ngram_range=(1, 3), min_df=1)
vectorizer, resume_matrix, normalized_resumes = build_resume_vector_space(resume_texts, vectorizer=vectorizer)
job_vector = transform_job_description(job_description, vectorizer)
scores = compute_similarity_scores(job_vector, resume_matrix)

shape_df = pd.DataFrame({
    'Object': ['Job Vector', 'Resume Matrix'],
    'Shape': [job_vector.shape, resume_matrix.shape],
})

scores_df = pd.DataFrame({
    'resume_index': list(range(len(normalized_resumes))),
    'similarity_score': scores,
    'resume_text': normalized_resumes,
}).sort_values(by='similarity_score', ascending=False).reset_index(drop=True)

display(shape_df)
display(scores_df.head(5))

C:\Users\POOJA\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


,Object,Shape
0,Job Vector,"(1, 9644)"
1,Resume Matrix,"(10, 9644)"


,resume_index,similarity_score,resume_text
0,3,0.027411,sou chef work experi sou chef jul compani ï¼​ ...
1,2,0.026551,digit market specialist summari digit market p...
2,4,0.024586,donor advoc profession summari organ professio...
3,0,0.021551,engin consult profession summari deliv valu pr...
4,5,0.012990,hr manag summari human resourc manag practic u...


## 3) Rank resumes using one-call retrieval

In [4]:
ranking_df, fitted_vectorizer, jd_vector, rs_matrix = rank_resumes_by_similarity(
    job_description=job_description,
    resume_texts=resume_texts,
    top_k=5,
)

print('Vectorizer:', fitted_vectorizer)
print('JD vector shape:', jd_vector.shape)
print('Resume matrix shape:', rs_matrix.shape)
display(ranking_df)

Vectorizer: TfidfVectorizer(dtype='float32', max_df=0.95, max_features=10000,
                ngram_range=(1, 3), stop_words='english',
                strip_accents='unicode', sublinear_tf=True)
JD vector shape: (1, 9644)
Resume matrix shape: (10, 9644)


C:\Users\POOJA\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


,resume_index,similarity_score,resume_text
0,3,0.027411,sou chef work experi sou chef jul compani ï¼​ ...
1,2,0.026551,digit market specialist summari digit market p...
2,4,0.024586,donor advoc profession summari organ professio...
3,0,0.021551,engin consult profession summari deliv valu pr...
4,5,0.012990,hr manag summari human resourc manag practic u...


## 4) Show JD terms and term-product explanation

In [5]:
feature_names = fitted_vectorizer.get_feature_names_out()
job_terms_df = get_nonzero_tfidf_terms(jd_vector, feature_names, value_column='jd_tfidf')
products_df = explain_term_products(
    job_vector=jd_vector,
    resume_matrix=rs_matrix,
    resume_texts=resume_texts,
    feature_names=feature_names,
    top_terms_per_resume=8,
)

display(job_terms_df.head(20))
display(products_df.head(40))

,term,jd_tfidf
0,data,1.0


,resume_index,resume_text,term,jd_tfidf,resume_tfidf,product
0,0,engin consult profession summari deliv valu pr...,data,1.0,0.021551,0.021551
1,2,digit market specialist summari digit market p...,data,1.0,0.026551,0.026551
2,3,sou chef work experi sou chef jul compani ï¼​ ...,data,1.0,0.027411,0.027411
3,4,donor advoc profession summari organ professio...,data,1.0,0.024586,0.024586
4,5,hr manag summari human resourc manag practic u...,data,1.0,0.012990,0.012990


## 5) Full retrieval report

In [6]:
report = retrieval_report(
    job_description=job_description,
    resume_texts=resume_texts,
    top_k=5,
    top_terms_per_resume=8,
)

print('Vectorizer:', report['vectorizer'])
print('Job vector shape:', report['job_vector_shape'])
print('Resume matrix shape:', report['resume_matrix_shape'])

display(report['ranking_df'])
display(report['job_terms_df'].head(20))
display(report['products_df'].head(40))

C:\Users\POOJA\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


Vectorizer: TfidfVectorizer(dtype='float32', max_df=0.95, max_features=10000,
                ngram_range=(1, 3), stop_words='english',
                strip_accents='unicode', sublinear_tf=True)
Job vector shape: (1, 9644)
Resume matrix shape: (10, 9644)


,resume_index,similarity_score,resume_text
0,3,0.027411,sou chef work experi sou chef jul compani ï¼​ ...
1,2,0.026551,digit market specialist summari digit market p...
2,4,0.024586,donor advoc profession summari organ professio...
3,0,0.021551,engin consult profession summari deliv valu pr...
4,5,0.012990,hr manag summari human resourc manag practic u...


,term,jd_tfidf
0,data,1.0


,resume_index,resume_text,term,jd_tfidf,resume_tfidf,product
0,0,engin consult profession summari deliv valu pr...,data,1.0,0.021551,0.021551
1,2,digit market specialist summari digit market p...,data,1.0,0.026551,0.026551
2,3,sou chef work experi sou chef jul compani ï¼​ ...,data,1.0,0.027411,0.027411
3,4,donor advoc profession summari organ professio...,data,1.0,0.024586,0.024586
4,5,hr manag summari human resourc manag practic u...,data,1.0,0.012990,0.012990
